# 1 — Data Download

Acquires the thesis dataset from **Binance public archives** (`https://data.binance.vision`) and builds the
aligned hourly panel consumed by the experiment. Pipeline:

1. **Download** monthly ZIP archives per asset and product (spot, futures and mark-price klines, plus funding rates), cached under `data/raw/binance`;
2. **Parse & merge** the archives into one CSV per asset and product (`BTCUSDT-futures-1h-merged.csv`, ...);
3. **Build the aligned panel** — one row per (hour, asset) with futures OHLCV, spot, mark price, funding event/last and basis — saved to `data/processed/`.

Each step is controlled by a switch in the configuration cell. Downloads are cached locally, so
re-running the notebook is fast once the archives have been fetched.

Assets: BTC, ETH, SOL, XRP (USDT perpetual futures + spot). Period: 2022-01-01 to 2026-04-30, hourly, UTC.

In [ ]:
from __future__ import annotations

import csv, io, json, time, zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib import error, parse, request

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

# ---------------------------------------------------------------------------
# All paths are anchored explicitly, so this notebook can run from anywhere.
# ---------------------------------------------------------------------------
# All data (ZIP cache, merged CSVs, aligned panel) lives under this folder.
DATA_DIR = Path('/Users/nunovieira/Masters/Thesis Final/data').resolve()

RAW_DIR        = DATA_DIR / 'raw' / 'binance'   # cached Binance ZIP archives
MERGED_CSV_DIR = DATA_DIR                        # per-asset merged CSVs
PROCESSED_DIR  = DATA_DIR / 'processed'          # aligned multi-asset panel
for folder in [RAW_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Scope of the dataset 
# ---------------------------------------------------------------------------
ASSETS   = ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'SOLUSDT']
START_TS = pd.Timestamp('2022-01-01 00:00:00+00:00')
END_TS   = pd.Timestamp('2026-04-30 23:00:00+00:00')
INTERVAL = '1h'

HOURLY_PRODUCTS = ['spot_klines', 'futures_klines', 'mark_price_klines']
OPTIONAL_HOURLY_PRODUCTS = ['premium_index_klines']  # referenced by helpers; not required
FUNDING_PRODUCT = 'funding_rate'

# ---------------------------------------------------------------------------
# Switches: one per pipeline step. Already-downloaded archives are never
# re-fetched, so a full re-run only re-parses and rebuilds.
# ---------------------------------------------------------------------------
RUN_DOWNLOAD              = True   # Step 1: fetch monthly ZIPs from data.binance.vision
EXPORT_CURRENT_STYLE_CSVS = True   # Step 2: re-create the merged CSVs from the ZIP cache
REBUILD_PANEL             = True   # Step 3: write the aligned panel to data/processed
OVERWRITE_DOWNLOADS       = False

print('Data folder:   ', DATA_DIR)
print('Target period: ', START_TS, '->', END_TS)
print('Switches:       RUN_DOWNLOAD =', RUN_DOWNLOAD,
      '| EXPORT_CURRENT_STYLE_CSVS =', EXPORT_CURRENT_STYLE_CSVS,
      '| REBUILD_PANEL =', REBUILD_PANEL)

Data folder:    /Users/nunovieira/Masters/Thesis Final/Code/data
Target period:  2022-01-01 00:00:00+00:00 -> 2026-04-30 23:00:00+00:00
Switches:       RUN_DOWNLOAD = True | EXPORT_CURRENT_STYLE_CSVS = True | REBUILD_PANEL = True


## Binance archive layout and parsing helpers

URL construction per product, ZIP download with a local cache, and parsers that normalise kline and funding files into tidy dataframes.

The parsers also handle the two raw-archive defects documented in Chapter 3 (and audited in
`2_DataExploration`): **embedded duplicate header rows** are detected and dropped, and **mixed
timestamp encodings** (13-digit millisecond vs 16-digit microsecond epochs) are unified by
`parse_binance_epoch`, which infers the resolution from the magnitude of the value.

In [ ]:
# Base URL for Binance's public historical data archive.
BASE_URL = 'https://data.binance.vision'

# PRODUCTS defines where each type of Binance data lives in the public archive.
# monthly is preferred because it downloads one ZIP per month instead of one ZIP per day.
# daily is kept here as a fallback pattern if a monthly archive is missing.
# kind tells the parser whether the file is normal OHLCV kline data or funding-rate event data.
PRODUCTS: dict[str, dict[str, str]] = {
    'spot_klines': {
        'monthly': 'data/spot/monthly/klines/{symbol}/{interval}/{symbol}-{interval}-{month}.zip',
        'daily': 'data/spot/daily/klines/{symbol}/{interval}/{symbol}-{interval}-{date}.zip',
        'kind': 'kline',
    },
    'futures_klines': {
        'monthly': 'data/futures/um/monthly/klines/{symbol}/{interval}/{symbol}-{interval}-{month}.zip',
        'daily': 'data/futures/um/daily/klines/{symbol}/{interval}/{symbol}-{interval}-{date}.zip',
        'kind': 'kline',
    },
    'mark_price_klines': {
        'monthly': 'data/futures/um/monthly/markPriceKlines/{symbol}/{interval}/{symbol}-{interval}-{month}.zip',
        'daily': 'data/futures/um/daily/markPriceKlines/{symbol}/{interval}/{symbol}-{interval}-{date}.zip',
        'kind': 'kline',
    },
    'premium_index_klines': {
        'monthly': 'data/futures/um/monthly/premiumIndexKlines/{symbol}/{interval}/{symbol}-{interval}-{month}.zip',
        'daily': 'data/futures/um/daily/premiumIndexKlines/{symbol}/{interval}/{symbol}-{interval}-{date}.zip',
        'kind': 'kline',
    },
    'funding_rate': {
        'monthly': 'data/futures/um/monthly/fundingRate/{symbol}/{symbol}-fundingRate-{month}.zip',
        'daily': 'data/futures/um/daily/fundingRate/{symbol}/{symbol}-fundingRate-{date}.zip',
        'kind': 'funding',
    },
}

# Binance kline files are CSVs without reliable headers; the expected column names are assigned manually.
KLINE_COLUMNS = [
    'open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_volume', 'trades', 'taker_buy_base_volume', 'taker_buy_quote_volume', 'ignore'
]

def month_starts(start: pd.Timestamp, end: pd.Timestamp) -> list[pd.Timestamp]:
    # Build one timestamp per month so that monthly Binance archives can be requested.
    return list(pd.date_range(start.normalize().replace(day=1), end.normalize().replace(day=1), freq='MS', tz='UTC'))

def month_label(ts: pd.Timestamp) -> str:
    # Convert a timestamp to the YYYY-MM string used in Binance monthly filenames.
    return ts.strftime('%Y-%m')

def archive_url(product: str, symbol: str, month: pd.Timestamp | None = None, date: pd.Timestamp | None = None) -> str:
    # Select the URL template for the requested product.
    spec = PRODUCTS[product]
    # Use a monthly URL when a month is provided.
    if month is not None:
        path = spec['monthly'].format(symbol=symbol, interval=INTERVAL, month=month_label(month))
    # Use a daily URL only when a specific date is provided.
    elif date is not None:
        path = spec['daily'].format(symbol=symbol, interval=INTERVAL, date=date.strftime('%Y-%m-%d'))
    else:
        raise ValueError('Provide month or date')
    # Combine the archive base URL with the product-specific path.
    return f'{BASE_URL}/{path}'

def local_archive_path(product: str, symbol: str, month: pd.Timestamp | None = None, date: pd.Timestamp | None = None) -> Path:
    # Build the same month/day label that appears in the remote ZIP filename.
    label = month_label(month) if month is not None else date.strftime('%Y-%m-%d')
    # Keep monthly and daily downloads in separate folders so they do not collide.
    scope = 'monthly' if month is not None else 'daily'
    # Extract just the remote ZIP filename from the full URL.
    name = Path(parse.urlparse(archive_url(product, symbol, month=month, date=date)).path).name
    # Cache the ZIP under data/raw/binance/<product>/<scope>/<symbol>/<year>/<file>.zip.
    return RAW_DIR / product / scope / symbol / label[:4] / name

def url_exists(url: str, timeout: int = 20) -> bool:
    # A HEAD request checks whether a file exists without downloading the full ZIP.
    req = request.Request(url, method='HEAD', headers={'User-Agent': 'Mozilla/5.0'})
    try:
        # If Binance responds with a 2xx or 3xx status, the archive is available.
        with request.urlopen(req, timeout=timeout) as response:
            return 200 <= response.status < 400
    except error.HTTPError as exc:
        # Some servers report redirects as HTTPError, so these codes are still treated as available.
        return exc.code in {200, 301, 302, 304}
    except Exception:
        # Timeouts, DNS errors, or missing files are treated as unavailable for the audit.
        return False

def download_url(url: str, dest: Path, overwrite: bool = False, sleep_s: float = 0.05) -> Path:
    # Reuse an existing local ZIP unless overwrite is explicitly requested.
    if dest.exists() and not overwrite:
        return dest
    # Make sure the destination folder exists before writing the file.
    dest.parent.mkdir(parents=True, exist_ok=True)
    # Download the remote ZIP bytes and save them locally.
    req = request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with request.urlopen(req, timeout=120) as response, open(dest, 'wb') as out:
        out.write(response.read())
    # Sleep briefly to avoid hammering the public archive with rapid requests.
    time.sleep(sleep_s)
    return dest

def parse_binance_epoch(series: pd.Series) -> pd.Series:
    # Convert the input to a Series so the function always returns a Series.
    # This keeps .iloc and .dt available in later cells.
    source = pd.Series(series)

    # Convert strings/numbers to numeric timestamps and coerce bad values to NaN.
    numeric = pd.to_numeric(source, errors='coerce')
    # Binance files usually use milliseconds, but this also handles microseconds/nanoseconds defensively.
    seconds = np.select(
        [numeric > 1e17, numeric > 1e14, numeric > 1e11],
        [numeric / 1e9, numeric / 1e6, numeric / 1e3],
        default=numeric,
    )
    # Return timezone-aware UTC timestamps for consistent merging and audits.
    # pd.to_datetime on a NumPy array returns a DatetimeIndex, so wrap it back into a Series.
    return pd.Series(pd.to_datetime(seconds, unit='s', utc=True), index=source.index)

def read_first_csv_from_zip(path: Path) -> pd.DataFrame:
    # Open the ZIP archive downloaded from Binance.
    with zipfile.ZipFile(path) as zf:
        # Find the CSV file inside the ZIP.
        csv_names = [name for name in zf.namelist() if name.endswith('.csv')]
        if not csv_names:
            raise ValueError(f'No CSV file inside {path}')
        # Read the first CSV without assuming it has headers.
        with zf.open(csv_names[0]) as fh:
            return pd.read_csv(fh, header=None)

def read_kline_zip(path: Path, product: str, symbol: str) -> pd.DataFrame:
    # Load one monthly/daily OHLCV ZIP into a dataframe.
    raw = read_first_csv_from_zip(path)
    # Some archives include a header row; if present, remove it before assigning standard names.
    if str(raw.iloc[0, 0]).lower() in {'open_time', 'opentime', 'time'}:
        raw = raw.iloc[1:].reset_index(drop=True)
    # Keep only the expected Binance kline columns.
    raw = raw.iloc[:, :min(raw.shape[1], len(KLINE_COLUMNS))].copy()
    raw.columns = KLINE_COLUMNS[:raw.shape[1]]
    # Add a parsed UTC timestamp for merging and missing-hour checks.
    raw['time'] = parse_binance_epoch(raw['open_time'])
    # Keep product/symbol metadata so combined dataframes remain traceable.
    raw['symbol'] = symbol
    raw['product'] = product
    # Convert numeric-looking columns from strings to actual numbers.
    numeric_cols = [c for c in raw.columns if c not in {'symbol', 'product', 'time'}]
    for col in numeric_cols:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')
    # Return the archive sorted by time.
    return raw.sort_values('time').reset_index(drop=True)

def read_funding_zip(path: Path, symbol: str) -> pd.DataFrame:
    # Load one Binance funding-rate ZIP into a dataframe.
    raw = read_first_csv_from_zip(path)
    # Check whether the first row is a header row.
    first_value = str(raw.iloc[0, 0]).lower()
    if first_value in {'calc_time', 'fundingtime', 'funding_time', 'time'}:
        # If a header row exists, use it as the dataframe columns and remove it from the data.
        headers = raw.iloc[0].astype(str).tolist()
        raw = raw.iloc[1:].reset_index(drop=True)
        raw.columns = headers
    else:
        # Otherwise assign the usual Binance funding columns manually.
        raw.columns = ['fundingTime', 'fundingRate', 'markPrice'][:raw.shape[1]]
    # Strip whitespace so column matching below is robust.
    raw.columns = [str(c).strip() for c in raw.columns]
    # Find the timestamp column even if Binance changes capitalization or naming slightly.
    time_col = next((c for c in raw.columns if c.lower() in {'calc_time', 'fundingtime', 'funding_time', 'time', 'timestamp'}), None)
    # Find the funding-rate column in the same defensive way.
    rate_col = next((c for c in raw.columns if c.lower() in {'last_funding_rate', 'fundingrate', 'funding_rate'}), None)
    if time_col is None or rate_col is None:
        raise ValueError(f'Could not identify funding time/rate columns in {path}: {list(raw.columns)}')
    # Normalize funding data to one row per funding event.
    out = pd.DataFrame({
        # Funding times are floored to the hour to match the 1h market files.
        'time': parse_binance_epoch(raw[time_col]).dt.floor('h'),
        'funding_rate_event': pd.to_numeric(raw[rate_col], errors='coerce'),
        'symbol': symbol,
        'product': 'funding_rate',
    })
    # Drop broken timestamps and return the events in chronological order.
    return out.dropna(subset=['time']).sort_values('time').reset_index(drop=True)

def target_months() -> list[pd.Timestamp]:
    # Convenience wrapper used by the probe, download, and load steps.
    return month_starts(START_TS, END_TS)


## Step 1 — Download monthly archives

Only runs when `RUN_DOWNLOAD = True`. Already-cached ZIPs are never re-downloaded.

In [3]:
# =====================================================================
# Step 1 -- Download monthly archives (only when RUN_DOWNLOAD = True)
# =====================================================================
def download_monthly_archives(products: list[str], symbols: list[str], availability_df: pd.DataFrame | None = None) -> list[Path]:
    # Store every downloaded or already-cached ZIP path here.
    downloaded = []
    # Loop through every requested product, asset, and target month.
    for product in products:
        for symbol in symbols:
            for month in target_months():
                month_text = month_label(month)
                # If the availability table says a ZIP does not exist, skip it instead of failing.
                if availability_df is not None and not availability_df.empty:
                    hit = availability_df.query('product == @product and symbol == @symbol and month == @month_text')
                    if len(hit) and not bool(hit.iloc[0]['exists']):
                        continue
                # Build the remote URL and local cache path for this one ZIP.
                url = archive_url(product, symbol, month=month)
                dest = local_archive_path(product, symbol, month=month)
                try:
                    # Download the ZIP unless it already exists and OVERWRITE_DOWNLOADS is False.
                    downloaded.append(download_url(url, dest, overwrite=OVERWRITE_DOWNLOADS))
                except Exception as exc:
                    # Keep going even if a single archive is missing or fails.
                    print(f'Missing or failed: {product} {symbol} {month_text}: {exc}')
    return downloaded

if RUN_DOWNLOAD:
    downloaded = download_monthly_archives(HOURLY_PRODUCTS + [FUNDING_PRODUCT], ASSETS)
    print('Archives downloaded or already cached:', len(downloaded))
else:
    print('Skipped (RUN_DOWNLOAD = False). The ZIP cache under data/raw/binance is used as-is.')

Archives downloaded or already cached: 832


## Step 2 — Parse cached ZIPs into merged per-asset CSVs

Only runs when `EXPORT_CURRENT_STYLE_CSVS = True`. Re-creates the 16 merged CSVs from the local ZIP cache.

In [4]:
# =====================================================================
# Step 2 -- Parse cached ZIPs into the per-asset merged CSVs
#           (only when EXPORT_CURRENT_STYLE_CSVS = True)
# =====================================================================
def load_cached_archives(products: list[str], symbols: list[str]) -> dict[str, dict[str, pd.DataFrame]]:
    # This nested dictionary will look like data[product][symbol] = dataframe.
    data: dict[str, dict[str, pd.DataFrame]] = {product: {} for product in products}
    # Read every ZIP that already exists in the local cache.
    for product in products:
        for symbol in symbols:
            frames = []
            for month in target_months():
                path = local_archive_path(product, symbol, month=month)
                if not path.exists():
                    continue
                try:
                    # Funding-rate files have different columns than kline OHLCV files.
                    if PRODUCTS[product]['kind'] == 'funding':
                        frame = read_funding_zip(path, symbol)
                    else:
                        frame = read_kline_zip(path, product, symbol)
                    frames.append(frame)
                except Exception as exc:
                    print(f'Failed to parse {path}: {exc}')
            if frames:
                # Combine all months for this product/symbol into one dataframe.
                combined = pd.concat(frames, ignore_index=True)
                # Trim to the requested research period.
                combined = combined[(combined['time'] >= START_TS) & (combined['time'] <= END_TS)].copy()
                # Sort after concatenation so later audit steps can assume chronological order.
                data[product][symbol] = combined.sort_values('time').reset_index(drop=True)
    return data


if EXPORT_CURRENT_STYLE_CSVS:
    archive_data = load_cached_archives(HOURLY_PRODUCTS + [FUNDING_PRODUCT], ASSETS)
    # Map the notebook's internal product names to the filename words used by the merged CSV files.
    # Example: futures_klines becomes BTCUSDT-futures-1h-merged.csv.
    PRODUCT_TO_CURRENT_SUFFIX = {
        'spot_klines': 'spot',
        'futures_klines': 'futures',
        'mark_price_klines': 'markprice',
    }
    
    def timestamps_to_milliseconds(times: pd.Series) -> pd.Series:
        """Convert timezone-aware pandas timestamps to Binance-style integer milliseconds."""
        # pandas stores datetimes internally as nanoseconds, so dividing by 1,000,000 gives milliseconds.
        return (pd.DatetimeIndex(times).astype('int64') // 1_000_000).astype('int64')
    
    def current_style_kline_frame(frame: pd.DataFrame) -> pd.DataFrame:
        """Return a kline dataframe with the merged-CSV column layout."""
        # Work on a copy so this export step never mutates the loaded raw dataframe.
        source = frame.copy()
    
        # Keep only rows inside the target period.
        source = source[(source['time'] >= START_TS) & (source['time'] <= END_TS)].copy()
    
        # Create Binance-style millisecond timestamps from the parsed hourly timestamp.
        open_time_ms = timestamps_to_milliseconds(source['time'])
    
        # Some Binance files include an ignore column and some derived files may not.
        # This keeps the output format consistent either way.
        ignore_values = source['ignore'] if 'ignore' in source.columns else pd.Series(0, index=source.index)
    
        # Recreate the exact column names used by the current files.
        out = pd.DataFrame({
            'open_time': open_time_ms,
            'open': pd.to_numeric(source['open'], errors='coerce'),
            'high': pd.to_numeric(source['high'], errors='coerce'),
            'low': pd.to_numeric(source['low'], errors='coerce'),
            'close': pd.to_numeric(source['close'], errors='coerce'),
            'volume': pd.to_numeric(source['volume'], errors='coerce'),
            'close_time': open_time_ms + (60 * 60 * 1000) - 1,
            'qav': pd.to_numeric(source['quote_volume'], errors='coerce'),
            'trades': pd.to_numeric(source['trades'], errors='coerce'),
            'tbbav': pd.to_numeric(source['taker_buy_base_volume'], errors='coerce'),
            'tbqav': pd.to_numeric(source['taker_buy_quote_volume'], errors='coerce'),
            'ignore': pd.to_numeric(ignore_values, errors='coerce').fillna(0),
        })
    
        # Sort chronologically and keep the last value if Binance ever provides a duplicate hour.
        out = out.sort_values('open_time').drop_duplicates(subset=['open_time'], keep='last').reset_index(drop=True)
    
        # Integer columns should stay integer in the saved CSV, matching the original thesis files.
        for col in ['open_time', 'close_time', 'trades', 'ignore']:
            out[col] = out[col].fillna(0).astype('int64')
    
        return out
    
    def current_style_funding_frame(frame: pd.DataFrame) -> pd.DataFrame:
        """Return a funding dataframe with the merged-CSV column layout."""
        # Work on a copy for the same reason as above: export should not mutate the loaded data.
        source = frame.copy()
    
        # Funding events are sparse (normally one every 8 hours), so only event timestamps are kept.
        source = source[(source['time'] >= START_TS) & (source['time'] <= END_TS)].copy()
    
        # Create the three-column funding format used by the current files.
        out = pd.DataFrame({
            'calc_time': timestamps_to_milliseconds(source['time']),
            'funding_interval_hours': 8,
            'last_funding_rate': pd.to_numeric(source['funding_rate_event'], errors='coerce'),
        })
    
        # Sort chronologically and remove duplicate funding timestamps if they appear.
        out = out.sort_values('calc_time').drop_duplicates(subset=['calc_time'], keep='last').reset_index(drop=True)
    
        # Keep timestamp and interval columns as integers in the saved file.
        out['calc_time'] = out['calc_time'].fillna(0).astype('int64')
        out['funding_interval_hours'] = out['funding_interval_hours'].astype('int64')
    
        return out
    
    def export_current_style_csvs(data: dict[str, dict[str, pd.DataFrame]]) -> pd.DataFrame:
        """Save one merged CSV per asset and product into MERGED_CSV_DIR."""
        export_rows = []
    
        # Export spot, futures, and mark-price hourly kline products.
        for product, suffix in PRODUCT_TO_CURRENT_SUFFIX.items():
            for symbol in ASSETS:
                frame = data.get(product, {}).get(symbol)
                if frame is None or frame.empty:
                    export_rows.append({'symbol': symbol, 'product': product, 'status': 'missing', 'path': None, 'rows': 0})
                    continue
    
                out = current_style_kline_frame(frame)
                path = MERGED_CSV_DIR / f'{symbol}-{suffix}-{INTERVAL}-merged.csv'
                out.to_csv(path, index=False)
                export_rows.append({'symbol': symbol, 'product': product, 'status': 'saved', 'path': str(path), 'rows': len(out)})
    
        # Export funding-rate event files in the merged-CSV format.
        for symbol in ASSETS:
            frame = data.get(FUNDING_PRODUCT, {}).get(symbol)
            if frame is None or frame.empty:
                export_rows.append({'symbol': symbol, 'product': FUNDING_PRODUCT, 'status': 'missing', 'path': None, 'rows': 0})
                continue
    
            out = current_style_funding_frame(frame)
            path = MERGED_CSV_DIR / f'{symbol}-funding-{INTERVAL}-merged.csv'
            out.to_csv(path, index=False)
            export_rows.append({'symbol': symbol, 'product': FUNDING_PRODUCT, 'status': 'saved', 'path': str(path), 'rows': len(out)})
    
        return pd.DataFrame(export_rows)
    
    if EXPORT_CURRENT_STYLE_CSVS:
        # Writes one merged CSV per asset and product into MERGED_CSV_DIR.
        current_style_exports = export_current_style_csvs(archive_data)
        if (current_style_exports['status'] == 'missing').all():
            print('No current-style CSVs were exported because no downloaded ZIP data has been loaded yet.')
            print('Set RUN_DOWNLOAD = True, rerun the download/load cell, then rerun this export cell.')
        display(current_style_exports)
    else:
        print('Set EXPORT_CURRENT_STYLE_CSVS = True to write the merged CSV files.')
else:
    print('Skipped (EXPORT_CURRENT_STYLE_CSVS = False). The 16 merged CSVs on disk are used as-is.')

,symbol,product,status,path,rows
0,BTCUSDT,spot_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37943
1,ETHUSDT,spot_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37943
2,XRPUSDT,spot_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37943
3,SOLUSDT,spot_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37943
4,BTCUSDT,futures_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37944
5,ETHUSDT,futures_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37944
6,XRPUSDT,futures_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37824
7,SOLUSDT,futures_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37824
8,BTCUSDT,mark_price_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37872
9,ETHUSDT,mark_price_klines,saved,/Users/nunovieira/Masters/Thesis Final/Code/da...,37896


## Step 3 — Build the aligned multi-asset panel

Loads the 16 merged CSVs, audits coverage, and assembles the panel per asset:
**futures klines and funding are the required feeds** (Chapter 3); spot and mark-price are merged as optional
left-joins (a handful of missing spot/mark hours become `NaN`). Funding events are merged onto the hourly grid
with missing event hours set to `0.0`, `funding_last` is the forward-filled last settled rate, and
`basis = log(mark/spot)`. No cross-asset intersection is applied: each asset keeps its own futures hours
(SOL and XRP have 120 fewer hours due to the 1--2 April 2022 listing gap).

In [5]:
# =====================================================================
# Step 3 -- Load the merged CSVs and build the aligned multi-asset panel
# =====================================================================
# The merged CSVs use compact Binance headers; rename to the standard names.
KLINE_RENAMES = {'qav': 'quote_volume', 'tbbav': 'taker_buy_base_volume', 'tbqav': 'taker_buy_quote_volume'}

def load_merged_kline(symbol: str, suffix: str) -> pd.DataFrame:
    frame = pd.read_csv(MERGED_CSV_DIR / f'{symbol}-{suffix}-{INTERVAL}-merged.csv').rename(columns=KLINE_RENAMES)
    frame['time'] = parse_binance_epoch(frame['open_time'])
    frame['symbol'] = symbol
    return frame

def load_merged_funding(symbol: str) -> pd.DataFrame:
    frame = pd.read_csv(MERGED_CSV_DIR / f'{symbol}-funding-{INTERVAL}-merged.csv')
    return pd.DataFrame({
        'time': parse_binance_epoch(frame['calc_time']).dt.floor('h'),
        'symbol': symbol,
        'funding_rate_event': pd.to_numeric(frame['last_funding_rate'], errors='coerce'),
    })

merged_data: dict[str, dict[str, pd.DataFrame]] = {p: {} for p in HOURLY_PRODUCTS + [FUNDING_PRODUCT]}
for symbol in ASSETS:
    merged_data['futures_klines'][symbol]    = load_merged_kline(symbol, 'futures')
    merged_data['spot_klines'][symbol]       = load_merged_kline(symbol, 'spot')
    merged_data['mark_price_klines'][symbol] = load_merged_kline(symbol, 'markprice')
    merged_data['funding_rate'][symbol]      = load_merged_funding(symbol)

# Coverage audit: rows, range, and duplicate timestamps per product and asset.
audit_rows = []
for p in HOURLY_PRODUCTS + [FUNDING_PRODUCT]:
    for s in ASSETS:
        f = merged_data[p][s]
        audit_rows.append({'product': p, 'symbol': s, 'rows': len(f),
                           'first': f['time'].min(), 'last': f['time'].max(),
                           'duplicates': int(f['time'].duplicated().sum())})
display(pd.DataFrame(audit_rows))

# Informational: the strict intersection of ALL hourly products across ALL assets.
# (Reported in the data-quality notebook; the panel itself only requires futures+funding.)
def common_hour_index(data, products, symbols) -> pd.DatetimeIndex:
    common = None
    for product in products:
        for symbol in symbols:
            idx = pd.DatetimeIndex(data[product][symbol]['time']).unique().sort_values()
            common = idx if common is None else common.intersection(idx)
    return common

strict_common = common_hour_index(merged_data, HOURLY_PRODUCTS, ASSETS)
print(f'Strict all-products/all-assets intersection (diagnostic only): {len(strict_common):,} hours')

def prepare_futures(frame: pd.DataFrame) -> pd.DataFrame:
    cols = ['time', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'quote_volume', 'trades', 'taker_buy_base_volume']
    out = frame[cols].copy()
    return out.rename(columns={
        'open': 'futures_open', 'high': 'futures_high', 'low': 'futures_low', 'close': 'futures_close',
        'volume': 'futures_volume', 'quote_volume': 'futures_quote_volume', 'trades': 'futures_trades',
        'taker_buy_base_volume': 'futures_taker_buy_volume',
    })

def prepare_spot(frame: pd.DataFrame) -> pd.DataFrame:
    cols = ['time', 'symbol', 'close', 'volume', 'quote_volume']
    out = frame[cols].copy()
    return out.rename(columns={'close': 'spot_close', 'volume': 'spot_volume', 'quote_volume': 'spot_quote_volume'})

def prepare_mark(frame: pd.DataFrame) -> pd.DataFrame:
    cols = ['time', 'symbol', 'close']
    out = frame[cols].copy()
    return out.rename(columns={'close': 'mark_close'})

def prepare_funding(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[['time', 'symbol', 'funding_rate_event']].copy()
    out['funding_rate_event'] = pd.to_numeric(out['funding_rate_event'], errors='coerce')
    return out.sort_values(['symbol', 'time'])

def build_aligned_panel(data: dict[str, dict[str, pd.DataFrame]]) -> pd.DataFrame:
    """Panel per Chapter 3: futures+funding required; spot/mark optional left-joins;
    no cross-asset hour intersection."""
    symbol_frames = []
    for symbol in ASSETS:
        futures = prepare_futures(data['futures_klines'][symbol])
        futures = futures[(futures['time'] >= START_TS) & (futures['time'] <= END_TS)].copy()
        spot = prepare_spot(data['spot_klines'][symbol])
        mark = prepare_mark(data['mark_price_klines'][symbol])
        merged = (futures
                  .merge(spot, on=['time', 'symbol'], how='left')
                  .merge(mark, on=['time', 'symbol'], how='left'))

        funding = prepare_funding(data['funding_rate'][symbol])
        merged = merged.merge(funding, on=['time', 'symbol'], how='left')
        merged['funding_rate_event'] = merged['funding_rate_event'].fillna(0.0)
        event_as_nan = merged['funding_rate_event'].replace(0.0, np.nan)
        merged['funding_last'] = event_as_nan.ffill().fillna(0.0)
        merged['basis'] = np.log(merged['mark_close'] / merged['spot_close'])
        symbol_frames.append(merged)

    panel = pd.concat(symbol_frames, ignore_index=True).sort_values(['time', 'symbol']).reset_index(drop=True)
    return panel

aligned_panel = build_aligned_panel(merged_data)
print('Panel rows:', len(aligned_panel), '| unique hours:', aligned_panel['time'].nunique())
display(aligned_panel.groupby('symbol')['time'].agg(['min', 'max', 'count']))

if REBUILD_PANEL:
    out_path = PROCESSED_DIR / f'aligned_panel_{START_TS:%Y%m%d}_{END_TS:%Y%m%d}.csv'
    aligned_panel.to_csv(out_path, index=False)
    print('Saved:', out_path)
else:
    print('Panel built in memory only (REBUILD_PANEL = False); no file was written.')

,product,symbol,rows,first,last,duplicates
0,spot_klines,BTCUSDT,37943,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
1,spot_klines,ETHUSDT,37943,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
2,spot_klines,XRPUSDT,37943,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
3,spot_klines,SOLUSDT,37943,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
4,futures_klines,BTCUSDT,37944,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
5,futures_klines,ETHUSDT,37944,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
6,futures_klines,XRPUSDT,37824,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
7,futures_klines,SOLUSDT,37824,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
8,mark_price_klines,BTCUSDT,37872,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0
9,mark_price_klines,ETHUSDT,37896,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,0


Strict all-products/all-assets intersection (diagnostic only): 37,751 hours
Panel rows: 151536 | unique hours: 37944


,min,max,count
symbol,,,
BTCUSDT,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,37944
ETHUSDT,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,37944
SOLUSDT,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,37824
XRPUSDT,2022-01-01 00:00:00+00:00,2026-04-30 23:00:00+00:00,37824


Saved: /Users/nunovieira/Masters/Thesis Final/Code/data/processed/aligned_panel_20220101_20260430.csv
